In [1]:
import pandas as pd
import numpy as np 
import sklearn
import networkx as nx
import ast

In [2]:
def centralities(edgelist):
    """
    - edgelist is a list of node pairs e.g. [(7,2),(1,7),(1,9),...]
    - returns a dictionary of vertex -> (centrality values)
    """
    T = nx.from_edgelist(edgelist)
    degree = dict(T.degree())
    avg_neighbor_deg = nx.average_neighbor_degree(T)
    degree_squared = {v: deg**2 for v, deg in dict(T.degree()).items()}
    degree_diff = {v: np.mean([T.degree(v) - T.degree(nbr) for nbr in T.neighbors(v)]) for v in T.nodes() }   
    clustering = nx.clustering(T)
    local_degree_ratio = { v: T.degree(v) / (np.mean([T.degree(nbr) for nbr in T.neighbors(v)]) + 1e-5) for v in T.nodes()}
    max_neighbor_degree = { v: max([T.degree(nbr) for nbr in T.neighbors(v)], default=0) for v in T.nodes() }
    dc = nx.degree_centrality(T)
    cc = nx.harmonic_centrality(T)
    bc = nx.betweenness_centrality(T)
    pc = nx.pagerank(T)
    # eigen_centrality = nx.eigenvector_centrality(T, max_)
    
    # return {v: (dc[v], cc[v], bc[v], pc[v] ) for v in T}
    return {
        v: (
            degree[v],
            avg_neighbor_deg[v],
            degree_squared[v],
            degree_diff[v],
            clustering[v],
            local_degree_ratio[v],
            max_neighbor_degree[v],
            dc[v],
            cc[v],
            bc[v],
            pc[v],
            # eigen_centrality[v]
        ) for v in T
    }

In [4]:
# Create train dataset
train = pd.read_csv('data/train.csv')

train_dataset = []
#iterate over the rows
for index, row in train.iterrows():
    edgelist_str = row['edgelist']
    edges = ast.literal_eval(edgelist_str)
    T = nx.from_edgelist(edges)
    features = centralities(edges)
    for node, centrality in features.items():
        root = 1 if node == row['root'] else 0
        sentence_id = row['sentence']
        language = row['language']
        length = len(T.nodes())

        train_dataset.append([sentence_id, language, node, 
                             centrality[0], centrality[1], centrality[2], 
                             centrality[3], centrality[4], centrality[5], 
                             centrality[6], centrality[7], centrality[8], 
                             centrality[9], centrality[10], root])


    
# convert to dataframe
train_dataset = pd.DataFrame(train_dataset, columns=['sentence_id', 'language', 'node',
                                                     'degree', 'avg_neighbor_deg', 'degree_squared', 
                                                     'degree_diff', 'clustering', 'local_degree_ratio',
                                                     'max_neighbor_degree', 'degree_centrality','harmonic_centrality', 
                                                     'betweenness_centrality', 'pagerank', 'root'])
# save to csv
train_dataset.to_csv('data/train_dataset_processed.csv', index=False)

In [7]:
# create test dataset
test = pd.read_csv('data/test.csv')
test_dataset = []
#iterate over the rows
for index, row in test.iterrows():
    edgelist_str = row['edgelist']
    edges = ast.literal_eval(edgelist_str)
    T = nx.from_edgelist(edges)
    features = centralities(edges)
    for node, centrality in features.items():
        # root = 1 if node == row['root'] else 0
        sentence_id = row['sentence']
        language = row['language']
        length = len(T.nodes())

        test_dataset.append([sentence_id, language, node, 
                             centrality[0], centrality[1], centrality[2], 
                             centrality[3], centrality[4], centrality[5], 
                             centrality[6], centrality[7], centrality[8], 
                             centrality[9], centrality[10]])


    
# convert to dataframe
test_dataset = pd.DataFrame(test_dataset, columns=['sentence_id', 'language', 'node',
                                                     'degree', 'avg_neighbor_deg', 'degree_squared', 
                                                     'degree_diff', 'clustering', 'local_degree_ratio',
                                                     'max_neighbor_degree', 'degree_centrality','harmonic_centrality', 
                                                     'betweenness_centrality', 'pagerank'])
# save to csv
test_dataset.to_csv('data/test_dataset_processed.csv', index=False)